In [1]:
from typing import Literal
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from pathlib import Path
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain.tools import tool
from langchain.chat_models import init_chat_model
import json


file_path = Path.cwd().parent / 'data' / 'cinderela.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    cind = f.read()

In [4]:
# ====== DEFINIÇÕES DAS 33 FUNÇÕES DE PROPP ======
PROPP_FUNCTIONS_DEFINITIONS = """
As 33 Funções Narrativas de Vladimir Propp:
I. α - Situação Inicial: Apresentação do tempo, lugar e dos membros da família do futuro herói.
II. β - Afastamento: Um dos membros da família (ou o futuro herói) afasta-se de casa (partida, morte, etc.).
III. γ - Interdição: Uma proibição ou ordem é dirigida ao herói.
IV. δ - Transgressão: A proibição é violada (conduzindo ao conflito).
V. ε - Interrogatório: O agressor tenta obter informações sobre a vítima.
VI. ζ - Obtenção de Informação: O agressor recebe informações sobre a vítima.
VII. η - Engano (Truque): O agressor tenta enganar a vítima (disfarce, persuasão).
VIII. θ - Cumplicidade: A vítima (ingenuamente ou por erro) deixa-se enganar e ajuda o agressor.
IX. A - Dano/Malefício: O agressor causa algum prejuízo ou dano a um membro da família ou ao herói (o problema central do conto).
X. a - Carência: Falta algo (dinheiro, objeto mágico, noiva) ao herói. (Alternativa a A).
XI. B - Mediação: O herói toma conhecimento do dano ou carência, é procurado ou chamado para a reparação, ou é autorizado a partir.
XII. C - Início da Ação Contrária: O herói aceita ou decide reagir (ir em busca, etc.).
XIII. ↑ - Partida: O herói deixa a casa para cumprir sua missão.
XIV. D - Primeira Função do Doador: O herói é posto à prova, interrogado, atacado, etc., preparando-o para receber o meio mágico.
XV. E - Reação do Herói: O herói reage à prova do doador, sendo bem ou mal-sucedido.
XVI. F - Recepção do Objeto Mágico: O herói adquire ou recebe um objeto, animal ou auxiliar mágico (ou conselho).
XVII. G - Deslocamento/Transferência: O herói é transportado para o local do objeto de busca ou do agressor.
XVIII. H - Luta (Combate): O herói e o agressor enfrentam-se em combate direto.
XIX. I - Marca: O herói recebe uma marca no corpo, cicatriz ou um objeto identificador.
XX. J - Vitória: O agressor é derrotado (morto, expulso, aprisionado).
XXI. K - Reparação: O dano ou a carência inicial é reparada (o objeto é recuperado, o feitiço é quebrado).
XXII. ↓ - Regresso: O herói volta para casa.
XXIII. Pr - Perseguição: O herói é perseguido por um inimigo.
XXIV. Rs - Socorro: O herói é salvo da perseguição (fuga, disfarce, intervenção do auxiliar).
XXV. O - Chegada Incógnita: O herói chega em casa ou a outro país sem ser reconhecido.
XXVI. L - Pretensões Falsas: Um falso herói tenta ocupar o lugar do herói verdadeiro.
XXVII. M - Tarefa Difícil: Uma tarefa difícil é imposta ao herói (para provar sua identidade ou obter algo).
XXVIII. N - Solução: A tarefa difícil é realizada.
XXIX. Ex - Reconhecimento: O herói verdadeiro é reconhecido (pela marca, pelo objeto identificador ou pela solução da tarefa).
XXX. T - Desmascaramento: O falso herói ou agressor é desmascarado.
XXXI. U - Transfiguração: O herói recebe nova aparência (é curado, fica mais belo, recebe roupas novas).
XXXII. W - Punição: O agressor ou falso herói é punido.
XXXIII. Q - Casamento: O herói casa-se (e/ou ascende ao trono).
IMPORTANTE: Nem todas as funções precisam estar presentes em um conto.
As funções devem ser identificadas na ordem em que aparecem na narrativa.
"""
# ====== ESTADO DO GRAFO ======
class GraphState(TypedDict):
    conto: str
    funcoes_identificadas: list[dict]
    sequencia_narrativa: str
    resultado_final: str
# ====== HELPER FUNCTIONS ======
def clean_json_response(content: str) -> str:
    """Remove markdown code blocks from JSON responses."""
    content = content.strip()
    # Remove markdown code blocks
    if content.startswith("```"):
        # Find the first newline after ```json or ```
        first_newline = content.find("\n")
        # Find the last ```
        last_backticks = content.rfind("```")
        if first_newline != -1 and last_backticks != -1:
            content = content[first_newline + 1 : last_backticks].strip()
    return content
# ====== INICIALIZAR MODELO ======
def obter_modelo():
    """Inicializa o modelo de forma limpa."""
    return init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
# ====== NÓS DO GRAFO ======
def node_identificar_funcoes(state: GraphState) -> GraphState:
    """Nó 1: Identifica as funções narrativas presentes no conto."""
    print("\n📖 Etapa 1: Identificando funções narrativas de Propp...")
    model = obter_modelo()
    prompt = f"""
Você é um especialista em Narratologia e na teoria de Vladimir Propp.
Sua tarefa é analisar contos e identificar quais das 33 funções narrativas de Propp estão presentes.
AS 33 FUNÇÕES DE PROPP:
{PROPP_FUNCTIONS_DEFINITIONS}
Conto a analisar:
{state["conto"]}
Identifique TODAS as funções presentes no conto, na ordem em que aparecem.
Para cada função identificada, forneça:
1. O símbolo da função (ex: α, β, A, etc.)
2. O nome da função
3. Uma breve citação ou descrição do trecho do conto que exemplifica essa função
4. Uma justificativa de por que essa função está presente
Responda com um JSON exatamente neste formato:
{{
    "funcoes": [
        {{
            "simbolo": "α",
            "nome": "Situação Inicial",
            "trecho": "Era uma vez uma menina...",
            "justificativa": "Apresenta os personagens e o cenário inicial"
        }},
        {{
            "simbolo": "γ",
            "nome": "Interdição",
            "trecho": "A mãe disse: não fale com estranhos",
            "justificativa": "Uma proibição é dirigida ao herói"
        }}
    ]
}}
Responda APENAS com o JSON, sem nenhum texto adicional.
"""
    response = model.invoke(prompt)
    cleaned_content = clean_json_response(response.content)
    try:
        result = json.loads(cleaned_content)
        state["funcoes_identificadas"] = result.get("funcoes", [])
    except json.JSONDecodeError as e:
        print(f"⚠️  Erro ao parsear JSON: {e}")
        print(f"📄 Resposta recebida (primeiros 500 chars):\n{response.content[:500]}")
        state["funcoes_identificadas"] = []
    print(f"✓ Funções identificadas: {len(state['funcoes_identificadas'])}")
    return state
def node_analisar_sequencia(state: GraphState) -> GraphState:
    """Nó 2: Analisa a sequência narrativa e padrões."""
    print("\n🔍 Etapa 2: Analisando sequência narrativa...")
    # Se não há funções identificadas, pular análise
    if not state["funcoes_identificadas"]:
        state["sequencia_narrativa"] = json.dumps(
            {
                "sequencia": "",
                "observacoes": "Nenhuma função foi identificada para análise",
                "tipo_conto": "Não classificado",
            },
            ensure_ascii=False,
            indent=2,
        )
        print("⚠️  Pulando análise - nenhuma função identificada")
        return state
    model = obter_modelo()
    funcoes_str = json.dumps(
        state["funcoes_identificadas"], ensure_ascii=False, indent=2
    )
    prompt = f"""
Você é um especialista em Narratologia.
Com base nas funções narrativas de Propp identificadas neste conto, faça uma análise da estrutura narrativa.
Funções identificadas:
{funcoes_str}
Forneça uma análise que inclua:
1. A sequência de símbolos das funções (ex: α-β-γ-δ-A-B-C-↑)
2. Observações sobre padrões narrativos (ex: presença de ciclos, funções ausentes importantes)
3. Classificação do tipo de conto baseado na estrutura (ex: conto de busca, conto de vitória sobre o agressor)
Responda com um JSON exatamente neste formato:
{{
    "sequencia": "α-β-γ-δ-A-B-C",
    "observacoes": "Este conto segue o padrão clássico...",
    "tipo_conto": "Conto de vitória sobre o agressor"
}}
Responda APENAS com o JSON, sem nenhum texto adicional.
"""
    response = model.invoke(prompt)
    cleaned_content = clean_json_response(response.content)
    try:
        result = json.loads(cleaned_content)
        state["sequencia_narrativa"] = json.dumps(result, ensure_ascii=False, indent=2)
    except json.JSONDecodeError as e:
        print(f"⚠️  Erro ao parsear JSON: {e}")
        print(f"📄 Resposta recebida (primeiros 500 chars):\n{response.content[:500]}")
        simbolos = [f["simbolo"] for f in state["funcoes_identificadas"]]
        state["sequencia_narrativa"] = json.dumps(
            {
                "sequencia": "-".join(simbolos),
                "observacoes": "Análise não disponível",
                "tipo_conto": "Não classificado",
            },
            ensure_ascii=False,
            indent=2,
        )
    print("✓ Análise de sequência concluída")
    return state
def node_formatar_resultado(state: GraphState) -> GraphState:
    """Nó 3: Formata o resultado final."""
    print("\n📊 Etapa 3: Formatando resultado...")
    resultado = "=" * 80 + "\n"
    resultado += "ANÁLISE NARRATOLÓGICA - 33 FUNÇÕES DE VLADIMIR PROPP\n"
    resultado += "=" * 80 + "\n\n"
    # Análise da sequência
    try:
        seq_data = json.loads(state["sequencia_narrativa"])
        resultado += "📈 ESTRUTURA NARRATIVA\n"
        resultado += "─" * 80 + "\n"
        resultado += f"   Sequência: {seq_data.get('sequencia', 'N/A')}\n"
        resultado += f"   Tipo de Conto: {seq_data.get('tipo_conto', 'N/A')}\n"
        resultado += f"   Observações: {seq_data.get('observacoes', 'N/A')}\n\n"
    except (json.JSONDecodeError, KeyError, TypeError):
        pass
    # Funções identificadas
    resultado += "📚 FUNÇÕES NARRATIVAS IDENTIFICADAS\n"
    resultado += "─" * 80 + "\n\n"
    for i, funcao in enumerate(state["funcoes_identificadas"], 1):
        resultado += f"{i}. {funcao.get('simbolo', '?')} - {funcao.get('nome', 'Desconhecida')}\n"
        resultado += f'   Trecho: "{funcao.get("trecho", "N/A")}"\n'
        resultado += f"   Justificativa: {funcao.get('justificativa', 'N/A')}\n\n"
    if not state["funcoes_identificadas"]:
        resultado += "   ℹ️  Nenhuma função narrativa foi identificada\n\n"
    state["resultado_final"] = resultado
    print("✓ Resultado formatado")
    return state
# ====== CONSTRUIR O GRAFO ======
def construir_grafo():
    """Constrói o grafo de análise narratológica."""
    graph = StateGraph(GraphState)
    # Adicionar nós
    graph.add_node("identificar_funcoes", node_identificar_funcoes)
    graph.add_node("analisar_sequencia", node_analisar_sequencia)
    graph.add_node("formatar_resultado", node_formatar_resultado)
    # Adicionar arestas (fluxo)
    graph.add_edge(START, "identificar_funcoes")
    graph.add_edge("identificar_funcoes", "analisar_sequencia")
    graph.add_edge("analisar_sequencia", "formatar_resultado")
    graph.add_edge("formatar_resultado", END)
    return graph.compile()

In [5]:
conto_chapeuzinho = """
Era uma vez uma menina chamada Chapeuzinho Vermelho que vivia com sua mãe 
numa cabana perto da floresta. Um dia, a mãe pediu que ela levasse uma cesta 
com comida para a avó que estava doente. A mãe advertiu: "Não saia do caminho 
e não fale com estranhos!"
No caminho, Chapeuzinho encontrou um lobo feroz que perguntou para onde ela ia. 
Ingenuamente, ela contou que estava levando comida para a avó e até indicou 
onde a avó morava. O lobo, astuto, sugeriu que ela colhesse flores para a avó.
Enquanto Chapeuzinho se distraía colhendo flores, o lobo correu para a casa 
da avó, a engoliu inteira e se disfarçou na cama dela. Quando Chapeuzinho chegou, 
conversou com o "lobo-avó" mas logo desconfiou das orelhas grandes, olhos grandes 
e boca grande. O lobo então tentou devorá-la.
Nesse momento, um caçador que passava pela floresta ouviu gritos, invadiu a casa 
e matou o lobo com seu machado, libertando a avó que ainda estava viva na barriga 
do lobo. Chapeuzinho e a avó agradeceram o caçador, e a menina aprendeu a nunca 
mais desobedecer sua mãe ou confiar em estranhos.
"""
# EXEMPLO 2: João e Maria
conto_joao_maria = """
João e Maria eram irmãos que viviam com o pai lenhador e a madrasta numa pequena 
casa na floresta. A família era muito pobre e não tinha comida suficiente. 
A madrasta convenceu o pai a abandonar as crianças na floresta para que não 
morressem todos de fome.
Na primeira vez, João ouviu o plano e deixou um rastro de pedrinhas brancas 
pelo caminho. Quando foram abandonados, conseguiram voltar para casa seguindo 
as pedras. Mas na segunda vez, João só tinha pão e deixou migalhas, que foram 
comidas pelos pássaros. As crianças ficaram perdidas na floresta.
Depois de dias vagando, encontraram uma casa feita de doces e pão. Famintos, 
começaram a comer a casa. De repente, uma bruxa velha apareceu e os convidou 
para entrar, mas era uma armadilha - ela queria engordar João para comê-lo.
A bruxa prendeu João numa gaiola e fez Maria trabalhar como escrava. Todos os 
dias a bruxa checava se João estava gordo o suficiente, mas ele espertamente 
mostrava um ossinho. Quando a bruxa perdeu a paciência e decidiu comer João 
de qualquer jeito, Maria a empurrou no forno aceso, matando-a.
As crianças encontraram tesouros na casa da bruxa, encheram os bolsos e 
conseguiram achar o caminho de volta para casa. Quando chegaram, descobriram 
que a madrasta havia morrido. O pai, que sempre sentiu falta deles, os recebeu 
com alegria. Com os tesouros da bruxa, nunca mais passaram fome.
"""
# EXEMPLO 3: Cinderela
conto_cinderela = """
Cinderela vivia com seu pai, madrasta e duas meias-irmãs. Depois que o pai 
morreu, a madrasta revelou sua verdadeira natureza cruel. Ela forçava Cinderela 
a fazer todo o trabalho doméstico enquanto suas filhas viviam no luxo.
Um dia, chegou um convite do palácio: o príncipe daria um grande baile para 
encontrar uma esposa. As meias-irmãs ficaram animadas, mas a madrasta proibiu 
Cinderela de ir, dando-lhe uma lista impossível de tarefas para completar.
Quando todos partiram para o baile, Cinderela chorou no jardim. Sua fada 
madrinha apareceu e, com magia, transformou uma abóbora em carruagem, ratos 
em cavalos, e os trapos de Cinderela num lindo vestido com sapatinhos de cristal. 
Mas advertiu: "A mágica acaba à meia-noite!"
No baile, o príncipe ficou encantado com Cinderela e dançou com ela a noite 
toda. Mas quando o relógio bateu meia-noite, ela fugiu correndo, perdendo um 
dos sapatinhos de cristal na escada.
O príncipe, determinado a encontrá-la, percorreu o reino fazendo todas as moças 
experimentarem o sapatinho. Quando chegou à casa de Cinderela, as meias-irmãs 
tentaram forçar o pé no sapato, mas não serviu. Cinderela pediu para tentar 
e o sapato se encaixou perfeitamente.
O príncipe reconheceu Cinderela como a misteriosa moça do baile. Eles se casaram 
e Cinderela, generosa, perdoou suas meias-irmãs e as convidou para viver no palácio.
"""
# Para testar, execute:
print("\n🚀 Iniciando análise narratológica...\n")
grafo = construir_grafo()
# Escolha qual conto testar:
conto_teste = conto_joao_maria  # ou conto_joao_maria ou conto_cinderela
resultado = grafo.invoke(
    {
        "conto": conto_teste,
        "funcoes_identificadas": [],
        "sequencia_narrativa": "",
        "resultado_final": "",
    }
)
# Exibir resultado formatado
print("\n" + resultado["resultado_final"])
# Exibir dados estruturados em JSON
print("\n📋 DADOS ESTRUTURADOS (JSON):")
print(
    json.dumps(
        {
            "funcoes_identificadas": resultado["funcoes_identificadas"],
            "sequencia_narrativa": json.loads(resultado["sequencia_narrativa"])
            if resultado["sequencia_narrativa"]
            else {},
        },
        ensure_ascii=False,
        indent=2,
    )
)


🚀 Iniciando análise narratológica...


📖 Etapa 1: Identificando funções narrativas de Propp...


/Users/gustavosarti/Work/code/xer/.venv/lib/python3.11/site-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


✓ Funções identificadas: 14

🔍 Etapa 2: Analisando sequência narrativa...
✓ Análise de sequência concluída

📊 Etapa 3: Formatando resultado...
✓ Resultado formatado

ANÁLISE NARRATOLÓGICA - 33 FUNÇÕES DE VLADIMIR PROPP

📈 ESTRUTURA NARRATIVA
────────────────────────────────────────────────────────────────────────────────
   Sequência: α-β-δ-A-B-C-η-IX-M-N-K-↓-O-Q
   Tipo de Conto: Conto de vitória sobre o agressor
   Observações: Este conto segue o padrão clássico de um herói enfrentando adversidades, com um ciclo de afastamento e retorno. As funções de 'ajuda' e 'recompensa' estão presentes, mas a função de 'prova' poderia ser mais desenvolvida. A presença de uma figura maligna (a bruxa) e a superação dela são centrais para a narrativa.

📚 FUNÇÕES NARRATIVAS IDENTIFICADAS
────────────────────────────────────────────────────────────────────────────────

1. α - Situação Inicial
   Trecho: "João e Maria eram irmãos que viviam com o pai lenhador e a madrasta numa pequena casa na floresta.